# Manual Colab UI: Full Mandarin Qwen3-ASR

This notebook avoids long `colab exec` / WebSocket automation. Run it manually in the Colab web UI.

Target episode: `mandarin_long_sushi` (`话说两宋三百年：苏轼`, ~116 min).

Design:

- Drive stores durable audio, model cache, logs, and outputs.
- `/content` is scratch only.
- ASR is checkpointed **after every chunk** into `qwen_chunks.jsonl`, so you can rerun/resume if Colab disconnects.
- Colab AI is optional text-only triage; it cannot listen to audio.


In [ ]:
#@title 0. Mount Drive and configure paths
from pathlib import Path
import os, sys, json, subprocess, time

from google.colab import drive
MOUNT_DRIVE = True  #@param {type:"boolean"}
if MOUNT_DRIVE:
    drive.mount('/content/drive')

PUBLIC_REPO_URL = "https://github.com/liuwen/qwen-asr-eval.git"  #@param {type:"string"}
PUBLIC_REPO_BRANCH = "main"  #@param {type:"string"}
REPO_DIR = Path("/content/qwen-asr-eval")  #@param {type:"string"}
DRIVE_ROOT = Path("/content/drive/MyDrive/asr")  #@param {type:"string"}
RUN_ID = "full_mandarin_long_sushi_manual"  #@param {type:"string"}

AUDIO_ROOT = DRIVE_ROOT / "audio"
RAW_AUDIO = AUDIO_ROOT / "raw_xiaoyuzhou" / "mandarin_long_sushi.audio"
HF_CACHE_ROOT = DRIVE_ROOT / "hf_cache"
RUN_DIR = DRIVE_ROOT / "qwen-asr-eval" / "runs" / RUN_ID
OUT_DIR = RUN_DIR / "outputs"
WORK_DIR = Path("/content/asr-eval-work") / RUN_ID
CHUNK_DIR = WORK_DIR / "chunks"

for p in [AUDIO_ROOT, HF_CACHE_ROOT, OUT_DIR, WORK_DIR, CHUNK_DIR]:
    p.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE_ROOT)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE_ROOT / "hub")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"

print("RUN_DIR:", RUN_DIR)
print("OUT_DIR:", OUT_DIR)
print("RAW_AUDIO:", RAW_AUDIO)
print("HF_HOME:", os.environ["HF_HOME"])


In [ ]:
#@title 1. Clone/update repo and install dependencies
import subprocess, sys, os
from pathlib import Path

if REPO_DIR.exists():
    print("Repo exists; pulling latest")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", PUBLIC_REPO_BRANCH], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", PUBLIC_REPO_BRANCH], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", PUBLIC_REPO_BRANCH, PUBLIC_REPO_URL, str(REPO_DIR)], check=True)

src_path = str(REPO_DIR / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg", "jq"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "qwen-asr", "huggingface_hub[hf_transfer]", "feedparser", "requests", "tqdm",
    "pandas", "numpy", "rapidfuzz", "soundfile", "librosa", "pydub", "jiwer",
    "opencc-python-reimplemented",
], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR), "--no-deps"], check=True)

# Optional: HF_TOKEN works in web UI if configured in Colab Secrets. Public model can also download without it.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    if HF_TOKEN:
        os.environ["HF_TOKEN"] = HF_TOKEN
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("HF_TOKEN configured from Colab Secrets.")
    else:
        print("HF_TOKEN not set; continuing with public model download.")
except Exception as e:
    print("HF_TOKEN unavailable; continuing if public download works:", repr(e))

print("ready")


In [ ]:
#@title 2. Ensure Mandarin source audio is present
from pathlib import Path
from asr_eval.xiaoyuzhou import resolve_xiaoyuzhou_audio_url, download_file
from asr_eval.audio import ffprobe_duration, fmt_ts

EPISODE_URL = "https://www.xiaoyuzhoufm.com/episode/69f08ec360313a2456c966c7"

if RAW_AUDIO.exists() and RAW_AUDIO.stat().st_size > 0:
    print("exists:", RAW_AUDIO, RAW_AUDIO.stat().st_size)
else:
    audio_url = resolve_xiaoyuzhou_audio_url(EPISODE_URL)
    print("resolved:", audio_url)
    download_file(audio_url, RAW_AUDIO)

print("duration:", fmt_ts(ffprobe_duration(RAW_AUDIO)))


In [ ]:
#@title 3. Resource check
import subprocess, shutil, json
from pathlib import Path

def run(cmd):
    return subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=False)

print("Disk:")
for p in ["/content", "/content/drive", str(DRIVE_ROOT)]:
    path = Path(p)
    if path.exists():
        u = shutil.disk_usage(path)
        print(json.dumps({"path": p, "used_gib": round(u.used/2**30,2), "free_gib": round(u.free/2**30,2), "total_gib": round(u.total/2**30,2)}))

print("
GPU:")
r = run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used,memory.free,utilization.gpu,temperature.gpu", "--format=csv,noheader,nounits"])
print((r.stdout or r.stderr).strip())

print("
GPU processes:")
r = run(["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory", "--format=csv,noheader,nounits"])
print((r.stdout or "<none>").strip())


In [ ]:
#@title 4. Normalize and chunk full audio
from asr_eval.audio import normalize_to_wav, chunk_wav, write_jsonl, ffprobe_duration, fmt_ts

CHUNK_SECONDS = 300  #@param {type:"integer"}
OVERLAP_SECONDS = 5  #@param {type:"integer"}
FORCE_REBUILD_CHUNKS = False  #@param {type:"boolean"}

NORMALIZED_WAV = WORK_DIR / "normalized_16k_mono.wav"
if FORCE_REBUILD_CHUNKS or not NORMALIZED_WAV.exists():
    NORMALIZED_WAV = normalize_to_wav(RAW_AUDIO, NORMALIZED_WAV)
else:
    print("exists:", NORMALIZED_WAV)

print("normalized duration:", fmt_ts(ffprobe_duration(NORMALIZED_WAV)))

manifest_path = OUT_DIR / "chunks_manifest.jsonl"
if FORCE_REBUILD_CHUNKS or not manifest_path.exists():
    chunks = chunk_wav(NORMALIZED_WAV, CHUNK_DIR, chunk_seconds=CHUNK_SECONDS, overlap_seconds=OVERLAP_SECONDS)
    write_jsonl(manifest_path, chunks)
else:
    chunks = []
    import json
    with manifest_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                chunks.append(json.loads(line))

print("chunks:", len(chunks))
print(json.dumps(chunks[:3], ensure_ascii=False, indent=2))
print("manifest:", manifest_path)


In [ ]:
#@title 5. Load Qwen3-ASR model
from asr_eval.qwen_runner import load_qwen_model

ASR_MODEL = "Qwen/Qwen3-ASR-1.7B"  #@param {type:"string"}
QWEN_MAX_INFERENCE_BATCH_SIZE = 1  #@param {type:"integer"}
QWEN_MAX_NEW_TOKENS = 4096  #@param {type:"integer"}

qwen_model = load_qwen_model(
    ASR_MODEL,
    max_inference_batch_size=QWEN_MAX_INFERENCE_BATCH_SIZE,
    max_new_tokens=QWEN_MAX_NEW_TOKENS,
    use_forced_aligner=False,
)
print("Loaded:", ASR_MODEL)


In [ ]:
#@title 6. Transcribe full audio with per-chunk checkpointing/resume
import json, time
from pathlib import Path
from asr_eval.qwen_runner import transcribe_chunks
from asr_eval.audio import write_jsonl
from asr_eval.reporting import save_transcript
import pandas as pd

QWEN_JSONL = OUT_DIR / "qwen_chunks.jsonl"
LANGUAGE = "Chinese"  #@param ["Chinese", "English", ""]
LANGUAGE = LANGUAGE or None
STOP_AFTER_N_NEW_CHUNKS = 0  #@param {type:"integer"}
# 0 means no artificial stop. Set e.g. 1 to test resume behavior.

existing_rows = []
completed_ids = set()
if QWEN_JSONL.exists():
    with QWEN_JSONL.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                existing_rows.append(row)
                completed_ids.add(int(row["chunk_id"]))
print("already completed chunks:", sorted(completed_ids))

new_done = 0
for chunk in chunks:
    cid = int(chunk["chunk_id"])
    if cid in completed_ids:
        continue
    print(f"
=== chunk {cid}/{len(chunks)-1}: {chunk['start_ts']} - {chunk['end_ts']} ===")
    t0 = time.time()
    rows = transcribe_chunks(
        qwen_model,
        [chunk],
        language=LANGUAGE,
        batch_size=1,
        return_time_stamps=False,
        model_name=ASR_MODEL,
    )
    # Append immediately for checkpoint/resume.
    with QWEN_JSONL.open("a", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "
")
            f.flush()
    existing_rows.extend(rows)
    completed_ids.add(cid)
    new_done += 1
    print(f"saved chunk {cid}; elapsed {time.time()-t0:.1f}s; text preview:")
    print(str(rows[0].get("text") or "")[:800])

    # Keep CSV/transcript usable even before entire run finishes.
    sorted_rows = sorted(existing_rows, key=lambda r: int(r["chunk_id"]))
    pd.DataFrame(sorted_rows).drop(columns=["time_stamps"], errors="ignore").to_csv(OUT_DIR / "qwen_chunks.csv", index=False)
    saved = save_transcript(OUT_DIR, "qwen", "Qwen3-ASR full Mandarin transcript", sorted_rows)
    print("checkpoint transcript:", saved["md_path"])

    if STOP_AFTER_N_NEW_CHUNKS and new_done >= STOP_AFTER_N_NEW_CHUNKS:
        print("STOP_AFTER_N_NEW_CHUNKS reached; rerun this cell to resume.")
        break

print("
completed", len(completed_ids), "of", len(chunks))
print("jsonl:", QWEN_JSONL)
print("md:", OUT_DIR / "qwen_transcript_chunked.md")
print("txt:", OUT_DIR / "qwen_transcript_chunked.txt")


In [ ]:
#@title 7. Rebuild transcript artifacts from qwen_chunks.jsonl
import json
import pandas as pd
from pathlib import Path
from asr_eval.reporting import save_transcript

QWEN_JSONL = OUT_DIR / "qwen_chunks.jsonl"
rows = []
with QWEN_JSONL.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))
rows = sorted(rows, key=lambda r: int(r["chunk_id"]))

pd.DataFrame(rows).drop(columns=["time_stamps"], errors="ignore").to_csv(OUT_DIR / "qwen_chunks.csv", index=False)
saved = save_transcript(OUT_DIR, "qwen", "Qwen3-ASR full Mandarin transcript", rows)
print(json.dumps({
    "rows": len(rows),
    "text_chars": sum(len(str(r.get("text") or "")) for r in rows),
    "first_range": [rows[0].get("start_ts"), rows[0].get("end_ts")],
    "last_range": [rows[-1].get("start_ts"), rows[-1].get("end_ts")],
    "csv": str(OUT_DIR / "qwen_chunks.csv"),
    "md": str(saved["md_path"]),
    "txt": str(saved["txt_path"]),
}, ensure_ascii=False, indent=2))


In [ ]:
#@title 8. Optional Colab AI text-only triage on selected chunks
RUN_COLAB_AI_TEXT_JUDGE = False  #@param {type:"boolean"}
COLAB_AI_MODEL = "google/gemini-3.5-flash"  #@param {type:"string"}
MAX_TEXT_JUDGE_CHUNKS = 6  #@param {type:"integer"}

if RUN_COLAB_AI_TEXT_JUDGE:
    from asr_eval.colab_ai_judge import judge_chunks_colab_ai
    reports = judge_chunks_colab_ai(rows, whisper_rows=None, use_case="zh", model_name=COLAB_AI_MODEL, max_chunks=MAX_TEXT_JUDGE_CHUNKS)
    judge_path = OUT_DIR / "colab_ai_text_judge.json"
    judge_path.write_text(json.dumps(reports, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Saved:", judge_path)
    import pandas as pd
    display(pd.DataFrame(reports))
else:
    print("Skipped. Set RUN_COLAB_AI_TEXT_JUDGE=True to run text-only triage.")
